# Word embeddings

One-hot vectors represent words as isolated points in a high-dimensional space. They carry no information about meaning: *king* and *queen* are as far apart as *king* and *sandwich*. Word embeddings fix this by replacing each word with a short dense vector learned from context. This notebook builds CBOW from scratch in NumPy, reimplements it in PyTorch, and then uses gensim's Word2Vec on a real corpus.

Order: intuition, maths, from-scratch version, PyTorch version, library version, visualisation, what breaks.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from gensim.models import Word2Vec
from nltk.corpus import gutenberg
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from torch import nn, optim

SEED = 42
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (7, 4)

## 1. Intuition

The **distributional hypothesis**: words that appear in similar contexts have similar meanings (Firth, 1957). *Bank* and *financial* often appear near *loan*, so they should sit close together in vector space. *Bank* and *river* share different neighbours, so they might be far from *financial*.

One-hot encoding ignores all of this. A vocabulary of size $V$ gives each word a vector of length $V$ with one 1 and $V-1$ zeros. The cosine similarity between any two distinct one-hot vectors is zero, so the representation is useless for measuring meaning.

In [ ]:
# Demonstrate the one-hot similarity problem
vocab_demo = ["king", "queen", "man", "woman", "sandwich"]
V = len(vocab_demo)
one_hot = np.eye(V)

sim = cosine_similarity(one_hot)
print("One-hot cosine similarities:")
for i, w in enumerate(vocab_demo):
    sims = [(vocab_demo[j], round(sim[i, j], 2)) for j in range(V) if j != i]
    print(f"  {w}: {sims}")

Every pair scores 0.0 — the model genuinely cannot tell that *king* and *queen* are more related than *king* and *sandwich*. A dense embedding learned from context will produce a very different picture.

## 2. Maths

**CBOW (Continuous Bag of Words):** given the $2k$ context words surrounding a target, predict the target word.

$$\hat{y} = \text{softmax}(W_2 \cdot \text{mean}(E[c_1], \ldots, E[c_{2k}]))$$

where $E \in \mathbb{R}^{V \times d}$ is the embedding matrix, $c_i$ are context word indices, and $W_2 \in \mathbb{R}^{V \times d}$ projects back to vocabulary size. Loss is cross-entropy over the vocabulary.

**Skip-gram:** inverts the objective — given the target word, predict each context word. Empirically better on small corpora.

**Negative sampling:** full softmax over a 50,000-word vocabulary requires 50,000 dot products per update. Negative sampling replaces it with a binary classifier: for one real context pair $(w, c)$ score high; for $k$ randomly drawn pairs $(w, c')$ score low. This reduces each update to $k+1$ dot products.

## 3. From scratch: CBOW in NumPy

Small corpus from the first 50 sentences of *Emma* (Gutenberg). Vocabulary capped at words that appear at least twice, giving roughly 150–200 unique words — enough to fit in seconds.

In [ ]:
# --- corpus ---
raw_sents = gutenberg.sents("austen-emma.txt")
clean_sents = []
for s in raw_sents[1:]:
    words = [w.lower() for w in s if w.isalpha()]
    if len(words) >= 3:
        clean_sents.append(words)
    if len(clean_sents) >= 50:
        break

# --- vocabulary ---
all_tokens = [w for s in clean_sents for w in s]
counts = Counter(all_tokens)
vocab = ["<PAD>"] + sorted(w for w, c in counts.items() if c >= 2)
w2i = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"Sentences: {len(clean_sents)}, tokens: {len(all_tokens)}, vocab: {V}")

In [ ]:
# --- CBOW training pairs ---
WINDOW = 2

def make_cbow_pairs(sents: list, w2i: dict, window: int) -> list:
    pairs = []
    for s in sents:
        idx = [w2i.get(w, 0) for w in s]
        for i in range(window, len(idx) - window):
            ctx = idx[i - window : i] + idx[i + 1 : i + window + 1]
            pairs.append((ctx, idx[i]))
    return pairs

pairs = make_cbow_pairs(clean_sents, w2i, WINDOW)
print(f"Training pairs: {len(pairs)}")

In [ ]:
# --- NumPy CBOW ---
EMB_DIM = 10
np.random.seed(SEED)

E = np.random.randn(V, EMB_DIM) * 0.01   # embedding matrix
W2 = np.random.randn(EMB_DIM, V) * 0.01  # output weights
b2 = np.zeros(V)


def softmax(x: np.ndarray) -> np.ndarray:
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()


def cbow_step(
    ctx: list, target: int, lr: float = 0.05
) -> float:
    global W2, b2
    # forward
    ctx_vecs = E[ctx]           # (2k, d)
    h = ctx_vecs.mean(axis=0)   # (d,)
    logits = h @ W2 + b2        # (V,)
    probs = softmax(logits)     # (V,)
    loss = -np.log(probs[target] + 1e-9)
    # backward
    dlogits = probs.copy()
    dlogits[target] -= 1.0
    W2 -= lr * np.outer(h, dlogits)
    b2 -= lr * dlogits
    dh = W2 @ dlogits
    for c in ctx:
        E[c] -= lr * dh / len(ctx)
    return loss


for epoch in range(5):
    total_loss = sum(cbow_step(ctx, tgt) for ctx, tgt in pairs)
    print(f"epoch {epoch + 1}  loss {total_loss / len(pairs):.4f}")

In [ ]:
# Nearest neighbours using the learned NumPy embedding
def nearest(word: str, E: np.ndarray, w2i: dict, vocab: list, n: int = 5) -> list:
    if word not in w2i:
        return []
    vec = E[w2i[word]].reshape(1, -1)
    sims = cosine_similarity(vec, E)[0]
    top = np.argsort(sims)[::-1][1 : n + 1]
    return [(vocab[i], round(float(sims[i]), 3)) for i in top]

for probe in ["emma", "man", "good"]:
    if probe in w2i:
        print(f"{probe}: {nearest(probe, E, w2i, vocab)}")

Loss falls with each epoch — the embedding matrix is learning. The neighbours make loose semantic sense even on 50 sentences.

## 4. CBOW in PyTorch

`nn.Embedding` replaces the index-into-matrix operation. PyTorch handles gradients automatically, so the backward pass disappears.

In [ ]:
class CBOWModel(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int) -> None:
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.proj = nn.Linear(emb_dim, vocab_size)

    def forward(self, ctx: torch.Tensor) -> torch.Tensor:
        # ctx: (batch, 2*window)
        h = self.emb(ctx).mean(dim=1)  # mean over context words
        return self.proj(h)


ctx_tensor = torch.tensor([p[0] for p in pairs], dtype=torch.long)
tgt_tensor = torch.tensor([p[1] for p in pairs], dtype=torch.long)

torch.manual_seed(SEED)
cbow_pt = CBOWModel(V, EMB_DIM)
optimiser = optim.Adam(cbow_pt.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

losses = []
for epoch in range(10):
    optimiser.zero_grad()
    logits = cbow_pt(ctx_tensor)
    loss = loss_fn(logits, tgt_tensor)
    loss.backward()
    optimiser.step()
    losses.append(loss.item())
    if (epoch + 1) % 2 == 0:
        print(f"epoch {epoch + 1:2d}  loss {loss.item():.4f}")

plt.plot(range(1, 11), losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("PyTorch CBOW training")
plt.tight_layout()
plt.show()

In [ ]:
# Extract the learned embedding matrix from PyTorch model
E_pt = cbow_pt.emb.weight.detach().numpy()  # (V, EMB_DIM)
print("Embedding matrix shape:", E_pt.shape)

for probe in ["emma", "man", "good"]:
    if probe in w2i:
        print(f"{probe}: {nearest(probe, E_pt, w2i, vocab)}")

Ten epochs on 50 sentences is not enough to learn rich semantics, but the loss falls consistently and the architecture is correct. The same model scales to a large corpus by streaming batches through a DataLoader.

## 5. Word2Vec with gensim

Gensim's Word2Vec trains skip-gram (or CBOW) with negative sampling on the full *Emma* corpus — 7,000+ sentences, far more than the scratch version. The result has actual semantic structure.

In [ ]:
emma_corpus = [
    [w.lower() for w in s if w.isalpha()]
    for s in gutenberg.sents("austen-emma.txt")
    if len(s) > 3
]

w2v = Word2Vec(
    emma_corpus,
    vector_size=50,
    window=3,
    min_count=5,
    workers=1,
    seed=SEED,
    epochs=10,
)
print(f"Vocabulary size: {len(w2v.wv)}")

In [ ]:
# Most similar words
for probe in ["emma", "mr", "love", "good"]:
    if probe in w2v.wv:
        neighbours = [(w, round(s, 3)) for w, s in w2v.wv.most_similar(probe, topn=5)]
        print(f"{probe:8s}: {neighbours}")

In [ ]:
# Analogy: mr - man + woman ≈ ?
# (Emma uses titles not generic gender words, so title analogies are more visible)
if all(w in w2v.wv for w in ["mr", "miss", "mrs"]):
    result = w2v.wv.most_similar(positive=["emma", "mr"], negative=["miss"], topn=5)
    print("emma + mr - miss:", [(w, round(s, 3)) for w, s in result])

# Cosine similarity between specific pairs
pairs_to_check = [("emma", "harriet"), ("emma", "knightley"), ("love", "marriage")]
for a, b in pairs_to_check:
    if a in w2v.wv and b in w2v.wv:
        sim = w2v.wv.similarity(a, b)
        print(f"sim({a}, {b}) = {sim:.3f}")

## 6. Visualisation

t-SNE projects the 50-dimensional embeddings onto a plane. Words that the model places nearby in embedding space should cluster together on the plot.

In [ ]:
# Top 60 words by frequency in the gensim model
top_words = [w for w, _ in w2v.wv.key_to_index.items()][:60]
top_vecs = np.array([w2v.wv[w] for w in top_words])

tsne = TSNE(n_components=2, random_state=SEED, perplexity=15, max_iter=1000)
pts = tsne.fit_transform(top_vecs)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(pts[:, 0], pts[:, 1], s=10, alpha=0.6)
for i, word in enumerate(top_words):
    ax.annotate(word, (pts[i, 0], pts[i, 1]), fontsize=7, alpha=0.8)
ax.set_title("t-SNE of top-60 Word2Vec embeddings (Emma)")
ax.axis("off")
plt.tight_layout()
plt.show()

## 7. What breaks

**Rare words.** Words seen fewer than `min_count` times get no vector. OOV at inference time needs a fallback (mean of subword pieces, or a zero vector).

**Polysemy.** *Bank* (river) and *bank* (financial) share one vector, averaged across all contexts where it appears. If the corpus is mixed the vector is blurry. Contextualised models (ELMo, BERT) produce a different vector for each occurrence.

**No subword handling.** *running* and *run* are unrelated in a plain Word2Vec. FastText fixes this by decomposing words into character n-grams, which also gives vectors to unseen morphological variants.

**Short context windows miss long-range dependencies.** A window of 3 captures local syntax but not discourse-level relationships. Transformers replace the fixed window with self-attention over the full sequence.